# Day 02 — ML performance (ROC AUC, confusion, so sánh ROI)

**Slides (tải về):** [`day02_slides.pptx`](_static/slides/day02_slides.pptx)

Mục tiêu:
- Chạy mô hình dự đoán EGFR trên demo dataset.
- Tính ROC AUC và confusion matrix.
- So sánh feature sets: clinical, intra, ring1, ring3, ring5.

Sản phẩm cuối buổi:
- Bảng AUC cho 5 feature sets.
- 1 hình ROC cho mô hình tốt nhất.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (6,4)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Nếu chạy trên Colab và không thấy file, notebook sẽ tải demo CSV từ GitHub.
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPO_NAME"
BRANCH = "main"

def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out_path = Path(rel_path).name
    print("Tải file:", url)
    urllib.request.urlretrieve(url, out_path)
    return Path(out_path)

def load_csv(rel_path: str, fallback_upload: bool = True) -> pd.DataFrame:
    candidates = [
        Path(rel_path),
        Path("../")/rel_path,
        Path("../../")/rel_path,
        Path("data")/Path(rel_path).name,
        Path("_static/data")/Path(rel_path).name,
        Path("../data")/Path(rel_path).name,
    ]
    for p in candidates:
        if p.exists():
            return pd.read_csv(p)

    # Thử tải từ GitHub
    try:
        p = download_from_github(rel_path)
        return pd.read_csv(p)
    except Exception as e:
        print("Không tải được từ GitHub:", e)

    # Cuối cùng: upload thủ công
    if fallback_upload:
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if len(uploaded) > 0:
                up_path = Path(list(uploaded.keys())[0])
                return pd.read_csv(up_path)
        except Exception:
            pass

    raise FileNotFoundError("Không tìm thấy CSV. Cần clone repo hoặc upload file.")

df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()


## 1) Tạo các feature sets

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

y = df["egfr_mutation"].astype(int)

clinical_cols = ["age", "sex", "smoking_status", "histology", "stage", "tumor_size_mm", "tumor_volume_cm3"]
intra_cols = [c for c in df.columns if c.startswith("intra_")]
ring1_cols = [c for c in df.columns if c.startswith("ring1_")]
ring3_cols = [c for c in df.columns if c.startswith("ring3_")]
ring5_cols = [c for c in df.columns if c.startswith("ring5_")]

feature_sets = {
    "clinical": clinical_cols,
    "intra": intra_cols,
    "ring1": ring1_cols,
    "ring3": ring3_cols,
    "ring5": ring5_cols,
}

{k: len(v) for k,v in feature_sets.items()}


## 2) Hàm train test và vẽ ROC

In [ ]:
def evaluate_feature_set(X: pd.DataFrame, y: pd.Series, name: str, random_state: int = 42):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=random_state, stratify=y
    )

    # Tự nhận biết cột numeric/categorical
    cat_cols = [c for c in X.columns if X[c].dtype == "object"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ],
        remainder="drop"
    )

    model = LogisticRegression(max_iter=2000)
    pipe = Pipeline(steps=[("pre", pre), ("model", model)])

    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, proba)
    fpr, tpr, _ = roc_curve(y_test, proba)

    # Confusion matrix tại threshold 0.5
    y_pred = (proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)

    return auc, (fpr, tpr), cm, pipe

# chạy thử 1 feature set
X_tmp = df[feature_sets["clinical"]]
auc, roc_xy, cm, pipe = evaluate_feature_set(X_tmp, y, "clinical")
auc, cm


## 3) So sánh AUC giữa các feature sets

In [ ]:
results = []
models = {}

for name, cols in feature_sets.items():
    X = df[cols].copy()
    auc, roc_xy, cm, pipe = evaluate_feature_set(X, y, name)
    results.append({"feature_set": name, "auc_test": auc})
    models[name] = {"roc": roc_xy, "cm": cm, "pipe": pipe}

res_df = pd.DataFrame(results).sort_values("auc_test", ascending=False)
res_df


## 4) Vẽ ROC cho feature set tốt nhất

In [ ]:
best_name = res_df.iloc[0]["feature_set"]
fpr, tpr = models[best_name]["roc"]
auc_best = res_df.iloc[0]["auc_test"]

plt.plot(fpr, tpr, label=f"{best_name} (AUC={auc_best:.3f})")
plt.plot([0,1],[0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()

print("Confusion matrix (threshold 0.5) -", best_name)
print(models[best_name]["cm"])


## Bài tập cuối buổi
1) Feature set nào tốt nhất theo AUC?
2) Confusion matrix: TP, FP, TN, FN là gì?
3) Nếu muốn ưu tiên bắt EGFR(+), threshold nên tăng hay giảm?
4) Viết 5 dòng: mô hình output là gì và AUC nói lên điều gì.